<b><h1 style="font-family: 'Helvetica Neue', Arial, sans-serif;">A assinatura linguística da fake news</h1><b>

<h3 style="font-family: 'Helvetica Neue', Arial, sans-serif;">Notícias falsas em português têm uma assinatura linguística, ou seja,
características de escrita que as distinguem das verdadeiras?</h3>

Conferindo que nenhum arquivo está faltando:

In [180]:
from pathlib import Path
import os
RAW = Path("../data/raw")
pastas = ["fake", "true", "fake-meta-information", "true-meta-information"]
for pasta in pastas:
    arquivos = os.listdir(RAW / pasta)
    print(pasta, len(arquivos))

fake 3600
true 3600
fake-meta-information 3600
true-meta-information 3600


Se cada pasta não tiver 3600 arquivos algo está errado

In [181]:
import pandas as pd

linhas = []
rotulos = {"fake": 1, "true": 0}
for rotulo in rotulos.keys():
    for arq in os.listdir(RAW / rotulo):
        arq = RAW / rotulo / arq
        linhas.append({
            "par_id": int(arq.stem),
            "rotulo": rotulos[rotulo],
            "texto": arq.read_text(encoding="utf-8"),
        })
textos = pd.DataFrame(linhas)

<b><h3>Montando a base</h3></b>

Cada texto nas pastas meta-information possui informações importantes, incluindo: a categoria do texto, numero de tokens no texto, emotividade, etc. Por isso é importante criar um Dataframe com essas informações.

In [182]:
colunas = ["autor", "link", "categoria", "data", "n_tokens", "n_palavras",
           "n_types", "n_links", "n_maiusculas", "n_verbos", "n_subj_imp",
           "n_substantivos", "n_adjetivos", "n_adverbios", "n_modais",
           "n_pron_sing_1_2", "n_pron_plur_1", "n_pronomes", "pausalidade",
           "n_caracteres", "tam_medio_sentenca", "tam_medio_palavra",
           "erros_ortograficos", "emotividade", "diversidade"]

linhas = []
for rotulo in rotulos.keys():
    for arq in os.listdir(RAW / f'{rotulo}-meta-information'):
        arq = RAW / f'{rotulo}-meta-information' / arq
        # Não usar .strip() no texto inteiro: alguns arquivos têm o autor vazio (" ") na
        # primeira linha, e o strip apagaria essa linha, deslocando todos os campos
        valores = [v.strip() for v in arq.read_text(encoding="utf-8").rstrip("\n").split("\n")]
        reg = dict(zip(colunas, valores))
        reg["par_id"] = int(arq.stem.split("-")[0])
        reg["rotulo"] = rotulos[rotulo]
        linhas.append(reg)
meta = pd.DataFrame(linhas)

In [183]:
base = textos.merge(meta, on=["par_id", "rotulo"], how="outer", indicator=True)
base = base.drop(columns=["_merge"])

int_columns = ["n_tokens", "n_palavras", "n_types", "n_links", 
               "n_maiusculas", "n_verbos", "n_subj_imp",
           "n_substantivos", "n_adjetivos", "n_adverbios", "n_modais",
           "n_pron_sing_1_2", "n_pron_plur_1", "n_pronomes"
]
float_columns = ["pausalidade", "n_caracteres", "tam_medio_sentenca", 
                 "tam_medio_palavra", "erros_ortograficos", 
                 "emotividade", "diversidade"
]

for column in int_columns:
    base[column] = base[column].astype("int", errors="ignore")

for column in float_columns:
    base[column] = base[column].astype("float", errors="ignore")

display(base.head())

,par_id,rotulo,texto,autor,link,categoria,data,n_tokens,n_palavras,n_types,...,n_pron_sing_1_2,n_pron_plur_1,n_pronomes,pausalidade,n_caracteres,tam_medio_sentenca,tam_medio_palavra,erros_ortograficos,emotividade,diversidade
0,1,0,﻿O Podemos decidiu expulsar o deputado federa...,Naira Trindade,http://politica.estadao.com.br/blogs/coluna-do...,politica,13/12/2017,168,148,107,...,0,0,7,3.33333,761.0,24.6667,5.14189,0.000000,0.134328,0.722973
1,1,1,Kátia Abreu diz que vai colocar sua expulsão e...,mrk,https://ceticismopolitico.com/2017/11/30/katia...,politica,2017-11-30,211,185,120,...,2,0,26,2.00000,815.0,14.2308,4.40541,0.000000,0.263158,0.648649
2,2,0,Em evento realizado nesta terça-feira para div...,Estadão Conteúdo,http://esportes.estadao.com.br/noticias/futebo...,sociedade_cotidiano,26/12/2017,349,294,182,...,0,0,29,2.75000,1477.0,14.7000,5.02381,0.000000,0.325758,0.619048
3,2,1,Blog esquerdista dá a entender que reclamar de...,None,https://ceticismopolitico.com/2017/11/28/blog-...,sociedade_cotidiano,2017-11-28,356,300,187,...,0,0,30,3.29412,1321.0,17.6471,4.40333,0.013333,0.277372,0.623333
4,3,0,﻿Após o prefeito de Manaus Arthur Virgílio (PS...,Juliana Diógenes,"http://politica.estadao.com.br/noticias/geral,...",politica,26/12/2017,249,204,128,...,3,0,8,3.00000,1019.0,13.6000,4.99510,0.014706,0.160000,0.627451


<b><h3>Checagens de qualidade</h3></b>

In [184]:
# Checa se tem algum valor nan
print(list(int(base[coluna].isna().sum()) for coluna in base.columns))

# Preenchendo os valores de diversidade Nan com a diversidade mediana daquela categoria
mediana = base.groupby(by="rotulo")["diversidade"].median()

base.loc[(base["diversidade"].isna()) & (base["rotulo"] == 0), "diversidade"] = mediana[0]
base.loc[(base["diversidade"].isna()) & (base["rotulo"] == 1), "diversidade"] = mediana[1]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [185]:
# Checagem de duplicados
texto = base[base.duplicated(subset="texto")]["texto"]
display(base[base["texto"] == texto.values[0]])

# Retirando texto duplicado
base = base.drop(index=120).reset_index()
display(base[base["texto"] == texto.values[0]])

,par_id,rotulo,texto,autor,link,categoria,data,n_tokens,n_palavras,n_types,...,n_pron_sing_1_2,n_pron_plur_1,n_pronomes,pausalidade,n_caracteres,tam_medio_sentenca,tam_medio_palavra,erros_ortograficos,emotividade,diversidade
120,61,0,Suplicy participará de programa de Doria na we...,Mônica Bergamo,http://www1.folha.uol.com.br/colunas/monicaber...,tv_celebridades,9/8/2017,160,136,97,...,0,0,4,4.0,614.0,22.6667,4.51471,0.0,0.109375,0.713235
136,69,0,Suplicy participará de programa de Doria na we...,None,http://g1.globo.com/jornal-nacional/noticia/20...,politica,7/10/2017,160,136,97,...,0,0,4,4.0,614.0,22.6667,4.51471,0.0,0.109375,0.713235


,index,par_id,rotulo,texto,autor,link,categoria,data,n_tokens,n_palavras,...,n_pron_sing_1_2,n_pron_plur_1,n_pronomes,pausalidade,n_caracteres,tam_medio_sentenca,tam_medio_palavra,erros_ortograficos,emotividade,diversidade
135,136,69,0,Suplicy participará de programa de Doria na we...,None,http://g1.globo.com/jornal-nacional/noticia/20...,politica,7/10/2017,160,136,...,0,0,4,4.0,614.0,22.6667,4.51471,0.0,0.109375,0.713235


In [186]:
# Encontrando textos vazios ou muito curtos
display(base.sort_values(by="n_palavras")["n_palavras"].head())

# Como tiramos um duplicado da base real iremos tirar o menor texto da base real
base = base.drop(base.sort_values(by="n_palavras")["n_palavras"].head(1).index)
print(len(base.loc[base["rotulo"] == 0]), len(base.loc[base["rotulo"] == 1]))

3208     9
3240     9
4062     9
6766     9
2482    10
Name: n_palavras, dtype: int64

3599 3599


In [187]:
tab = pd.crosstab(base.categoria, base.rotulo)
display(base.categoria.unique())
display(tab)

<StringArray>
[           'politica', 'sociedade_cotidiano',     'tv_celebridades',
  'ciencia_tecnologia',            'religiao',            'economia']
Length: 6, dtype: str

rotulo,0,1
categoria,,
ciencia_tecnologia,56,56
economia,22,22
politica,2090,2089
religiao,22,22
sociedade_cotidiano,638,638
tv_celebridades,771,772


Para deixar a base mais simétrica iremos tirar o texto com menor numero de linha falso de tv_celebridades e o texto com menor numero de linhas real de politica

In [188]:
base = base.drop(index=base.loc[(base["categoria"] == "politica") & (base["rotulo"] == 0)] \
                 .sort_values(by=["n_palavras"], ascending=True)["n_palavras"].head(1).index)
base = base.drop(index=base.loc[(base["categoria"] == "tv_celebridades") & (base["rotulo"] == 1)] \
                 .sort_values(by=["n_palavras"], ascending=True)["n_palavras"].head(1).index)

tab = pd.crosstab(base.categoria, base.rotulo)
display(base.categoria.unique())
display(tab)

<StringArray>
[           'politica', 'sociedade_cotidiano',     'tv_celebridades',
  'ciencia_tecnologia',            'religiao',            'economia']
Length: 6, dtype: str

rotulo,0,1
categoria,,
ciencia_tecnologia,56,56
economia,22,22
politica,2089,2089
religiao,22,22
sociedade_cotidiano,638,638
tv_celebridades,771,771


In [189]:
base.to_csv("../data/processed/fakebr_base.csv", index=False)